# Notebook 1: Lexical vs Semantic Search

A basic overview of lexical (keyword-based text searches) and semantic search.

Goal: Understand the difference between lexical search and semantic search. 

In [5]:
from pathlib import Path
import sys

sys.path.append("../")

# Check imports
from src.config import REPO

## Lexical vs Semantic — what is the difference?

**Lexical search** matches *the literal text* of the query — either as raw
substrings or as tokens.<br><br>

The query "train" matches documents that contain the
literal substring `train` (e.g., "trains", "trained", etc.).  Under the hood each document is represented as a *sparse*
vector — one dimension per vocabulary token, almost all entries zero.<br><br><br>

**Semantic search** matches *meaning*. Each document is projected onto a dense
vector (" word embedding") produced by a language model.<br><br>
Search  computes the cosine similarity between the query vector and *each* document vector. Documents are ranked by that score in descending order. Because the model has learned associations between synonyms, paraphrases, and related concepts during training, documents with related meanings cluster together in the vector space — so a semantic search query for "train" can surface a post about "the new high-speed rail line.

### Why lexical alone is not enough

The query `train` will only match posts that literally contain the substring
`train` (or `training`, `trainer`, etc. via tokenisation). It will *not* match a post
about a *high-speed rail line*, a *subway commute*, or a *locomotive*. BM25 (the
Elasticsearch-backed `KeywordSearcher`) is smarter about term frequency and document
length, but it is still fundamentally token matching — synonyms are invisible.

Semantic search closes that gap by comparing *embeddings* of the query and document
instead of their tokens.<br><br><br>

# What is an embedding?

An embedding is generated by a machine learning model that has been trained to
understand the relationships between words and phrases in a large corpus of
text. The resulting embedding is a vector of numbers that defines the position
of a piece of text relative to the corpus the model was trained on as a whole.


Link: Understanding Emeddings

## What is: cosine similarity? 

The search queries above depend on KNN (k-nearest neighbors) cosine similarity
to rank results, 

Cosine similarity is the cosine of the angle between two vectors, normalized. Two
points pointing in the same direction from the origin score 1.0; perpendicular points
score 0.0.<br><br>

<img src="images/image_cosine_similarity.png" alt="Vector similarity illustration" style="width: 100%; max-width: 100%; height: auto;" />

<br><br><br>




## Cosine similarity in practice<br>


In [6]:
import numpy as np
import pandas as pd
from IPython.display import Math, display

display(Math(
    r"\text{Cosine Similarity} = "
    r"\frac{\sum_{i=1}^{n} A_i B_i}"
    r"{\sqrt{\sum_{i=1}^{n} A_i^2}\, \sqrt{\sum_{i=1}^{n} B_i^2}}"
))

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine similarity between two vectors: dot product / (|a| * |b|)."""
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def show_cosine_math(a: np.ndarray, b: np.ndarray) -> None:
    numerator_terms = [int(x * y) for x, y in zip(a, b, strict=True)]
    a_sq_terms = [int(x * x) for x in a]
    b_sq_terms = [int(y * y) for y in b]

    numerator = int(np.dot(a, b))
    norm_a = float(np.sqrt(np.sum(np.square(a))) or 1)
    norm_b = float(np.sqrt(np.sum(np.square(b))) or 1)
    score = cosine_similarity(a, b)

    print("Dot Product (A · B)\n")
    print(f"Numerator: {' + '.join(map(str, numerator_terms))} = {numerator}\n")
    print("Denominator: \n")
    print(f"||A|| = sqrt({' + '.join(map(str, a_sq_terms))}) = {norm_a:.4f}")
    print(f"||B|| = sqrt({' + '.join(map(str, b_sq_terms))}) = {norm_b:.4f}\n")
    print(f"||A|| × ||B|| = {norm_a:.4f} × {norm_b:.4f} = {norm_a * norm_b:.4f}\n")
    print(f"==> Cosine similarity = {numerator} / ({norm_a:.4f} * {norm_b:.4f}) = {score:.4f}\n")


print("\n\nExample 1")
vocab1 = ["Hello", "World"]
A1 = np.array([1, 1])  # A = "Hello, World!"
B1 = np.array([1, 0])  # B = "Hello!"

df1 = pd.DataFrame(
    [A1, B1],
    index=["A = Hello, World!", "B = Hello!"],
    columns=vocab1,
)
display(df1)
show_cosine_math(A1, B1)

print("\n\nExample 2 (matches the larger token table idea):")
vocab2 = ["Python", "snake", "also", "programming", "language"]
A2 = np.array([1, 1, 0, 0, 0])  # A = "A Python is a snake"
B2 = np.array([1, 0, 1, 1, 1])  # B = "Python is also a programming language"

df2 = pd.DataFrame(
    [A2, B2],
    index=["A = A Python is a snake", "B = Python is also a programming language"],
    columns=vocab2,
)
display(df2)
show_cosine_math(A2, B2)
print("  Same cosine formula each time; only the vector length changes with each input.")


<IPython.core.display.Math object>



Example 1


,Hello,World
"A = Hello, World!",1,1
B = Hello!,1,0


Dot Product (A · B)

Numerator: 1 + 0 = 1

Denominator: 

||A|| = sqrt(1 + 1) = 1.4142
||B|| = sqrt(1 + 0) = 1.0000

||A|| × ||B|| = 1.4142 × 1.0000 = 1.4142

==> Cosine similarity = 1 / (1.4142 * 1.0000) = 0.7071



Example 2 (matches the larger token table idea):


,Python,snake,also,programming,language
A = A Python is a snake,1,1,0,0,0
B = Python is also a programming language,1,0,1,1,1


Dot Product (A · B)

Numerator: 1 + 0 + 0 + 0 + 0 = 1

Denominator: 

||A|| = sqrt(1 + 1 + 0 + 0 + 0) = 1.4142
||B|| = sqrt(1 + 0 + 1 + 1 + 1) = 2.0000

||A|| × ||B|| = 1.4142 × 2.0000 = 2.8284

==> Cosine similarity = 1 / (1.4142 * 2.0000) = 0.3536

  Same cosine formula each time; only the vector length changes with each input.


<br><b>NOTES:</b><br>
*KNN* (k-nearest neighbors) uses cosine similarity to identify the most similar vectors in a vector database.
There are over 30,000 words in the English language. It would be computationally expensive to run this math over vectors of that size. <br>

To address this we use a model trained on language that can identify word associations and other patterns
in text and return a "dense" vector. 
<br><br>

### 1. Lexical search query
Run a lexical search over a handful of real posts using
`InMemoryKeywordSearcher` from `src/search.py`. This searcher does a simple
case-insensitive substring count — it is the simplest possible lexical search.

In [7]:
### Example of a basic lexical search.
import json

from src.config import REPO
from src.search import InMemoryKeywordSearcher

# REPO points at the repo root regardless of the jupyter cwd
with open(REPO / "sample_posts.json") as f:
    posts = json.load(f)

print(f"Loaded {len(posts)} posts. Showing first 3:")
for post in posts[:3]:
    print(f"  • {post['post_text'][:80]}…")

# Run a lexical search for the literal token "train"
searcher = InMemoryKeywordSearcher(posts)
results = searcher.search("train", top_k=5)

print(f"\nLexical search for 'train' returned {len(results)} matches:")
for result in results:
    print(f"  score={result['score']}  →  {result['post_text'][:100]}…")

Loaded 154 posts. Showing first 3:
  • 🐾 Today's Caturday is extra special because we're celebrating Whiskers' birthday…
  • 🐱 Kitty can be such a joy, but they sure do have their quirks too. Like preferri…
  • 👩‍👧‍👦 Catmom and Dad share equally in the responsibilities of raising a litter o…

Lexical search for 'train' returned 5 matches:
  score=1  →  Federal funding for California's High Speed Rail project has increased significantly over recent yea…
  score=1  →  Advocates argue that bullet trains are essential for reducing traffic jams and air pollution. With f…
  score=1  →  The transition from swimming in pools to open water can be daunting, but it's a rewarding experience…
  score=1  →  For those training for open water competitions in cold climates like Canada’s Niagara, gear choice b…
  score=1  →  Preparing for open water races involves more than just physical training; it's about understanding y…


Elasticsearch uses a more sophisticated lexical search, but the principle is
the same: it looks for literal token matches.

In [ ]:
# Elasticsearch "BM25" (Best Matching 25), a common algorithm for ranking search results.
import inspect

from src.search import KeywordSearcher

print("── KeywordSearcher.search_similar_documents ──")
print(inspect.getsource(KeywordSearcher.search_similar_documents))

── KeywordSearcher.search_similar_documents ──
    def search_similar_documents(
        self, query: str, top_k: int = TOP_K_DEFAULT, filters: list[dict] | None = None
    ) -> list[dict]:
        """Run a BM25 match query against post_text and return ranked results."""
        body = {
            "size": top_k,
            "query": {"match": {"post_text": query}},
        }
        try:
            resp = self.client.search(index=self.index_name, body=body)
        except Exception as e:
            raise ConnectionError("Check that the docker container is running") from e
        results = [
            {
                "score": hit["_score"],
                **{k: v for k, v in hit["_source"].items() if k != "doc_embedding"},
            }
            for hit in resp["hits"]["hits"]
        ]
        return sorted(results, key=lambda result: result.get("score", 0.0), reverse=True)



### 2. Semantic search query

In [ ]:
### Example of a basic semantic search.
import json

from src.config import REPO
from src.search import InMemorySemanticSearcher

# REPO points at the repo root regardless of the jupyter cwd
with open(REPO / "sample_posts.json") as f:
    posts = json.load(f)

print(f"Loaded {len(posts)} posts. Showing first 3:")
for post in posts[:3]:
    print(f"  • {post['post_text'][:80]}…")

# Run a lexical search for the literal token "train"
searcher = InMemorySemanticSearcher(posts)
results = searcher.search("train", top_k=5)

print(f"\nSemantic search for 'train' returned {len(results)} matches:")
for result in results:
    print(f"  score={result['score']}  →  {result['post_text'][:100]}…")

INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: mps
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: all-MiniLM-L6-v2


Loaded 154 posts. Showing first 3:
  • 🐾 Today's Caturday is extra special because we're celebrating Whiskers' birthday…
  • 🐱 Kitty can be such a joy, but they sure do have their quirks too. Like preferri…
  • 👩‍👧‍👦 Catmom and Dad share equally in the responsibilities of raising a litter o…


KeyError: 'doc_embedding'

In [ ]:

# Under the hood both the InMemorySemanticSearcher and ElasticSearch take dense vectors and compute cosine similarity,

import inspect

from src.search import SemanticSearcher, InMemorySemanticSearcher

# This method mimics the Elasticsearch approach but is implemented in-memory for demonstration purposes.
print("── InMemorySemanticSearcher.search_similar_documents ──")
print(inspect.getsource(InMemorySemanticSearcher.search_similar_documents))

# This method uses Elasticsearch
print("── SemanticSearcher.search_similar_documents ──")
print(inspect.getsource(SemanticSearcher.search_similar_documents))

# Exercise: 

Get to know the data
Time: 3 minutes  

Before we move on to generating embeddings and building a topic model, it's helpful to get familiar with
the dataset.

Open [sample_posts.json](../sample_posts.json). 

```
REPO / sample_posts.json
```

- Identify three queries that would work in semantic search but not in lexical
  search.
- Why didn't the last three posts in the cell "2. Semantic Search Query" above display? 